# Week 2, Day 6: Forge: Week 2 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w002/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", (np.zeros((3, 1, 5)) + np.zeros((4, 1))).shape)
x = np.arange(6); y = x[::2]; y[0] = 99
print("2.", x[0])
print("3. mean", 5 * 2, "variance", 5 * 2)

In [ ]:
import numpy as np
rng = np.random.default_rng(2026)
n_neurons, n_ori, n_rep, window = 60, 12, 25, 0.5
oris = np.linspace(0, np.pi, n_ori, endpoint=False)                 # (12,)
true_pref = rng.uniform(0, np.pi, n_neurons)                         # (60,)
gain = rng.uniform(10, 40, n_neurons)
base = rng.uniform(1, 5, n_neurons)
kappa = 2.0

# Trial order: each orientation 25 times, shuffled
stim = rng.permutation(np.repeat(np.arange(n_ori), n_rep))          # (300,) orientation index per trial
rate = base[:, None] + gain[:, None] * np.exp(kappa * (np.cos(2 * (oris[stim][None, :] - true_pref[:, None])) - 1))
counts = rng.poisson(rate * window)                                  # (60, 300)
print("counts:", counts.shape, " stim:", stim.shape)

### Exercise: Part A: tuning curves

Compute `tuning` (shape 60 by 12): each neuron's mean firing rate in Hz for each orientation. Trials are shuffled, so you need masks. A loop over the 12 orientations is allowed; no loop over neurons or trials.

In [ ]:
tuning = ...
print(tuning.shape)

In [ ]:
import numpy as np
def test_shape():
    "tuning has shape (neurons, orientations)"
    assert tuning.shape == (60, 12)

def test_values():
    "tuning[i, j] is neuron i's mean rate (Hz) on trials with orientation j"
    j = 3
    assert np.allclose(tuning[:, j], counts[:, stim == j].mean(axis=1) / window)

test_shape()
test_values()
print("All checks passed")

### Exercise: Part B: preferred orientations

Estimate each neuron's preferred orientation as the orientation with the highest mean rate, using `np.argmax`. Store it in radians in `est_pref`. Then compare with `true_pref`: the median error should be under 10 degrees (the grid spacing is 15 degrees).

In [ ]:
est_pref = ...
err = np.abs(np.angle(np.exp(2j * (est_pref - true_pref)))) / 2   # orientation difference, wrapping at 180 degrees
print("median error:", np.degrees(np.median(err)).round(1), "degrees")

In [ ]:
import numpy as np
def test_shape():
    "est_pref has one orientation (radians) per neuron"
    assert est_pref.shape == (60,)

def test_error():
    "median error is under 10 degrees"
    err = np.abs(np.angle(np.exp(2j * (est_pref - true_pref)))) / 2
    assert np.degrees(np.median(err)) < 10, f"median error {np.degrees(np.median(err)):.1f} degrees"

test_shape()
test_error()
print("All checks passed")

### Exercise: Part C: is the population Poisson?

For every neuron and orientation, compute the Fano factor of the counts across the 25 repeats (variance with $n-1$ over mean). Store the (60, 12) array in `fanos` and print its median. Then plot a histogram.

In [ ]:
import matplotlib.pyplot as plt
fanos = ...
print("median Fano factor:", np.nanmedian(fanos).round(3))
plt.figure(figsize=(4, 3)); plt.hist(fanos.ravel(), bins=30); plt.xlabel("Fano factor"); plt.ylabel("count"); plt.show()

In [ ]:
import numpy as np
def test_fano_shape():
    "fanos has shape (neurons, orientations)"
    assert fanos.shape == (60, 12)

def test_fano_value():
    "the median Fano factor is close to 1"
    assert 0.85 < np.nanmedian(fanos) < 1.15

test_fano_shape()
test_fano_value()
print("All checks passed")

In [ ]:
import matplotlib.pyplot as plt
order = np.argsort(est_pref)
z = (tuning - tuning.mean(axis=1, keepdims=True)) / tuning.std(axis=1, keepdims=True)
plt.figure(figsize=(5, 4))
plt.imshow(z[order], aspect="auto", extent=[0, 180, 60, 0], cmap="magma")
plt.xlabel("orientation (deg)"); plt.ylabel("neuron (sorted)"); plt.colorbar(label="z-scored rate")
plt.show()